# YOLOv8 Waste Classification Training - Google Colab
## Train YOLOv8 on GPU for waste detection (Metal, Plastic, Paper)

**Setup Instructions:**
1. Upload this notebook to Google Colab
2. Upload your Combined_Dataset folder to Google Drive
3. Update the dataset path in Cell 2 if needed
4. Run all cells from top to bottom

## Cell 1: Mount Google Drive and Install Dependencies

In [ ]:
# Mount Google Drive (optional - only if using Google Drive)
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully!")

In [ ]:
# Install YOLOv8 and dependencies
!pip install -q ultralytics
!pip install -q torch torchvision

print("✓ All dependencies installed!")

## Cell 2: Configure Dataset Path

In [ ]:
import os
from pathlib import Path
import zipfile

# ===== IF UPLOADING AS ZIP FILE =====
# Uncomment and run this if you uploaded Combined Dataset.zip
zip_path = '/content/Combined Dataset.zip'
if os.path.exists(zip_path):
    print("Extracting ZIP file...")
    zipfile.ZipFile(zip_path).extractall('/content/')
    print("✓ ZIP extracted successfully!")

# ===== CHOOSE ONE OPTION BELOW =====

# OPTION 1: Direct upload to Colab (uncomment if using direct upload)
dataset_path = '/content/Combined Dataset'

# OPTION 2: Use Google Drive (uncomment if using Google Drive)
# dataset_path = '/content/drive/My Drive/Combined Dataset'

# ===== END OPTIONS =====

data_yaml = os.path.join(dataset_path, 'data.yaml')

# Verify dataset exists
if os.path.exists(data_yaml):
    print(f"✓ Dataset found at: {dataset_path}")
    print(f"✓ data.yaml path: {data_yaml}")
else:
    print(f"✗ Dataset NOT found at {dataset_path}")
    print("Please:")
    print("  1. Update the dataset_path variable above to match your setup")
    print("  2. Or upload your 'Combined Dataset' folder/ZIP file")

# List contents
print(f"\nDataset contents:")
try:
    for item in os.listdir(dataset_path):
        print(f"  - {item}")
except:
    print("  (folder not found or not uploaded yet)")

## Cell 3: Verify GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU is ready for training!")
else:
    print("\n✗ No GPU available. Training will be slow.")

## Cell 4: Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO

print("="*60)
print("YOLOv8 Training Configuration")
print("="*60)
print(f"Dataset YAML: {data_yaml}")
print(f"Model: YOLOv8 Medium")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"Epochs: 100")
print(f"Batch Size: 32")
print(f"Image Size: 640")
print("="*60)

# Load pretrained YOLOv8 model
model = YOLO('yolov8m.pt')

# Train the model
print("\nStarting training... (this will take 1-2 hours on GPU)\n")

results = model.train(
    data=data_yaml,
    epochs=100,              # Number of epochs
    imgsz=640,              # Image size
    batch=32,               # Batch size (GPU can handle larger batches)
    patience=20,            # Early stopping patience
    device=0,               # Use first GPU (auto-detects)
    workers=4,              # Number of dataloader workers
    save=True,              # Save model checkpoints
    verbose=True,           # Print training progress
    project='/content/drive/My Drive',  # Save results to Google Drive
    name='yolo_waste_detector',  # Run name
)

print("\n" + "="*60)
print("Training Complete!")
print("="*60)

## Cell 5: Verify Training Results

In [ ]:
import os

# Find the training results directory
results_dir = '/content/drive/My Drive/yolo_waste_detector'

if os.path.exists(results_dir):
    print(f"✓ Training results saved to: {results_dir}")
    
    # List contents
    for item in os.listdir(results_dir):
        print(f"  - {item}")
    
    # Check for best model
    best_model_path = os.path.join(results_dir, 'weights', 'best.pt')
    if os.path.exists(best_model_path):
        print(f"\n✓ Best model saved at: {best_model_path}")
        print(f"  File size: {os.path.getsize(best_model_path) / 1e6:.2f} MB")
else:
    print(f"✗ Results directory not found at {results_dir}")

## Cell 6: Test Model on Sample Image

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Load the trained model
best_model_path = '/content/drive/My Drive/yolo_waste_detector/weights/best.pt'
model = YOLO(best_model_path)

# Find a sample image from validation set
sample_image_dir = dataset_path + '/valid/images'
sample_images = [f for f in os.listdir(sample_image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

if sample_images:
    sample_image_path = os.path.join(sample_image_dir, sample_images[0])
    print(f"Testing on: {sample_images[0]}")
    
    # Run inference
    results = model.predict(source=sample_image_path, conf=0.25)
    
    # Display results
    for result in results:
        print(f"\nDetections found:")
        for box in result.boxes:
            cls_name = result.names[int(box.cls)]
            confidence = box.conf.item()
            print(f"  - {cls_name}: {confidence:.2%} confidence")
        
        # Display image with detections
        result_image = result.plot()
        plt.figure(figsize=(12, 8))
        plt.imshow(result_image[..., ::-1])  # Convert BGR to RGB
        plt.axis('off')
        plt.title('YOLOv8 Waste Detection Results')
        plt.tight_layout()
        plt.show()
else:
    print("No sample images found in validation set")

## Cell 7: Download Trained Model

The trained model is automatically saved to your Google Drive in the `yolo_waste_detector` folder. You can download it from there.

In [ ]:
# Display download instructions
print("="*60)
print("DOWNLOAD TRAINED MODEL")
print("="*60)
print("\nYour trained model is saved in Google Drive at:")
print("  /yolo_waste_detector/weights/best.pt")
print("\nTo download:")
print("  1. Open Google Drive")
print("  2. Navigate to yolo_waste_detector folder")
print("  3. Right-click on weights folder")
print("  4. Select 'Download'")
print("\nTo use the model in your local Python:")
print("  from ultralytics import YOLO")
print("  model = YOLO('path/to/best.pt')")
print("  results = model.predict(source='image.jpg')")
print("="*60)